In [43]:
import numpy as np
from sklearn.linear_model import LinearRegression

import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.utils.data.dataset import random_split
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('fivethirtyeight')

## data generation

In [44]:
true_b = 1
true_w = 2
N = 100

# Data Generation
np.random.seed(42)
x = np.random.rand(N, 1)
y = true_b + true_w * x + (.1 * np.random.randn(N, 1))

# Shuffles the indices
idx = np.arange(N)
np.random.shuffle(idx)

# Uses first 80 random indices for train
train_idx = idx[:int(N*.8)]
# Uses the remaining indices for validation
val_idx = idx[int(N*.8):]

# Generates train and validation sets
x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]

## data preparation

In [45]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Our data was in Numpy arrays, but we need to transform them
# into PyTorch's Tensors and then we send them to the 
# chosen device
x_train_tensor = torch.as_tensor(x_train).float().to(device)
y_train_tensor = torch.as_tensor(y_train).float().to(device)

## model config

In [46]:

# This is redundant now, but it won't be when we introduce
# Datasets...
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Sets learning rate - this is "eta" ~ the "n"-like Greek letter
lr = 0.1

torch.manual_seed(42)
# Now we can create a model and send it at once to the device
model = nn.Sequential(nn.Linear(1, 1)).to(device)

# Defines a SGD optimizer to update the parameters 
# (now retrieved directly from the model)
optimizer = optim.SGD(model.parameters(), lr=lr)

# Defines a MSE loss function
loss_fn = nn.MSELoss(reduction='mean')

In [47]:
n_epochs = 1000
for epoch in range(n_epochs):
    model.train()
    yhat = model(x_train_tensor)
    loss = loss_fn(yhat, y_train_tensor)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

In [48]:
print(model.state_dict())

OrderedDict([('0.weight', tensor([[1.9690]])), ('0.bias', tensor([1.0235]))])


In [49]:
def make_train_step(model, loss_fn, optimizer):
    def perfrom_train_step_fn(x, y):
        model.train()
        yhat = model(x)
        loss = loss_fn(yhat, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        return loss.item()
    return perfrom_train_step_fn

In [50]:

# This is redundant now, but it won't be when we introduce
# Datasets...
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Sets learning rate - this is "eta" ~ the "n"-like Greek letter
lr = 0.1

torch.manual_seed(42)
# Now we can create a model and send it at once to the device
model = nn.Sequential(nn.Linear(1, 1)).to(device)

# Defines a SGD optimizer to update the parameters 
# (now retrieved directly from the model)
optimizer = optim.SGD(model.parameters(), lr=lr)

# Defines a MSE loss function
loss_fn = nn.MSELoss(reduction='mean')

# create the train_step function for our model, loss function and optimizer
train_step_fn = make_train_step(model, loss_fn, optimizer)

In [51]:
train_step_fn

<function __main__.make_train_step.<locals>.perfrom_train_step_fn(x, y)>

In [52]:
n_epochs = 1000
losses = []

for epoch in range(n_epochs):
    loss = train_step_fn(x_train_tensor, y_train_tensor)
    losses.append(loss)

In [53]:
print(model.state_dict())

OrderedDict([('0.weight', tensor([[1.9690]])), ('0.bias', tensor([1.0235]))])


# Dataset

In [54]:
class CustomDataset(Dataset):
    def __init__(self, x_tensor, y_tensor):
        self.x = x_tensor
        self.y = y_tensor

    def __getitem__(self, index):
        return (self.x[index], self.y[index])

    def __len__(self):
        return len(self.x)

x_train_tensor = torch.as_tensor(x_train).float()
y_train_tensor = torch.as_tensor(y_train).float()

train_data = CustomDataset(x_train_tensor, y_train_tensor)
print(train_data[0])

(tensor([0.7713]), tensor([2.4745]))


## TensorDataset

In [55]:
train_data = TensorDataset(x_train_tensor, y_train_tensor)
print(train_data[0])

(tensor([0.7713]), tensor([2.4745]))


# DataLoader

In [56]:
train_loader = DataLoader(
    dataset=train_data,
    batch_size=16,
    shuffle=True,
)

In [57]:
next(iter(train_loader))

[tensor([[0.2809],
         [0.3253],
         [0.1560],
         [0.5924],
         [0.0651],
         [0.8872],
         [0.4938],
         [0.0055],
         [0.1409],
         [0.0885],
         [0.1849],
         [0.7290],
         [0.8662],
         [0.3117],
         [0.6842],
         [0.1987]]),
 tensor([[1.5846],
         [1.8057],
         [1.2901],
         [2.1687],
         [1.1559],
         [2.8708],
         [1.9060],
         [1.0632],
         [1.1211],
         [1.0708],
         [1.5888],
         [2.4927],
         [2.6805],
         [1.7637],
         [2.3492],
         [1.2654]])]

In [58]:
import os
os.makedirs("data_preparation", exist_ok=True)

In [59]:
%%writefile data_preparation/v1.py

x_train_tensor = torch.as_tensor(x_train).float()
y_train_tensor = torch.as_tensor(y_train).float()

train_data = TensorDataset(x_train_tensor, y_train_tensor)

train_loader = DataLoader(
    dataset=train_data,
    batch_size=16,
    shuffle=True
)

Overwriting data_preparation/v1.py


In [60]:
%run -i data_preparation/v1.py

In [61]:
os.makedirs('model_configuration', exist_ok=True)

In [62]:
%%writefile model_configuration/v2.py
n_epochs = 1000

losses = []

for epoch in range(n_epochs):
    mini_batch_losses = []
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        mini_batch_loss = train_step_fn(x_batch, y_batch)
        mini_batch_losses.append(mini_batch_loss)
    loss = np.mean(mini_batch_losses)
    losses.append(loss)

Overwriting model_configuration/v2.py


In [63]:
%run -i model_configuration/v2.py

In [64]:
print(model.state_dict())

OrderedDict([('0.weight', tensor([[1.9698]])), ('0.bias', tensor([1.0246]))])


## Mini-Batch Inner Loop

In [65]:
def mini_batch(device, data_loader, step_fn):
    mini_batch_losses = []
    for x_batch, y_batch in data_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        mini_batch_loss = step_fn(x_batch, y_batch)
        mini_batch_losses.append(mini_batch_loss)
    loss = np.mean(mini_batch_losses)
    return loss

In [66]:
os.makedirs('model_training', exist_ok=True)

In [67]:
%%writefile model_training/v3.py
n_epochs = 200
losses = []

for epoch in range(n_epochs):
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)

Overwriting model_training/v3.py


In [68]:
%run -i model_training/v3.py

In [69]:
print(model.state_dict())

OrderedDict([('0.weight', tensor([[1.9682]])), ('0.bias', tensor([1.0231]))])


In [70]:
%%writefile data_preparation/v2.py
torch.manual_seed(42)

x_tensor = torch.as_tensor(x).float()
y_tensor = torch.as_tensor(y).float()

dataset = TensorDataset(x_tensor, y_tensor)

# perfroms the split
ratio = .8
n_total = len(dataset)
n_train = int(n_total * ratio)
n_val = n_total - n_train
train_data, val_data = random_split(dataset, [n_train, n_val])

# buils a loader of each set
train_loader = DataLoader(
    dataset=train_data,
    batch_size=16,
    shuffle=True
)
val_loader = DataLoader(dataset=val_data, batch_size=16)

Overwriting data_preparation/v2.py


In [71]:
run -i data_preparation/v2.py

# Evalution

In [72]:
def make_val_step_fn(model, loss_fn):
    def perfrom_val_step_fn(x, y):
        # sets model to eval mode
        model.eval()
        yhat = model(x)
        loss = loss_fn(yhat, y)
        return loss.item()
    return perfrom_val_step_fn

In [73]:
%%writefile model_configuration/v2.py

device = 'cuda' if torch.cuda.is_available() else 'cpu'

lr = 0.1
torch.manual_seed(42)

model = nn.Sequential(nn.Linear(1, 1)).to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)
loss_fn = nn.MSELoss(reduction='mean')
train_step_fn = make_train_step(model, loss_fn, optimizer)
val_step_fn = make_val_step_fn(model, loss_fn)

Overwriting model_configuration/v2.py


In [74]:
%run -i model_configuration/v2.py

In [75]:
%%writefile model_training/v4.py
n_epochs = 200

losses = []
val_losses = []

for epoch in range(n_epochs):
    # inner loop
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)

    # validation - no gradients in validation
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
        val_losses.append(val_loss)

Overwriting model_training/v4.py


In [76]:
%run -i model_training/v4.py

In [77]:
print(model.state_dict())

OrderedDict([('0.weight', tensor([[1.9599]])), ('0.bias', tensor([1.0151]))])


# TensorBoard

In [78]:
# load the TensorBoard notebook extention
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [79]:
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 7086), started 0:00:02 ago. (Use '!kill 7086' to kill it.)

## summaryWriter

In [95]:
writer = SummaryWriter('runs/test')

## add_graph

In [96]:
dummy_x, dummy_y = next(iter(train_loader))
writer.add_graph(model, dummy_x.to(device))

In [97]:
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 7086), started 0:04:50 ago. (Use '!kill 7086' to kill it.)

## add_scalars

In [98]:
writer.add_scalars(
    main_tag='loss',
    tag_scalar_dict={'training': loss.item(),
                    'validation': val_loss.item()},
    global_step=epoch
)
writer.flush()

In [100]:
%tensorboard --logdir runs/test

In [101]:
%run -i data_preparation/v2.py

In [104]:
%%writefile model_configuration/v3.py
device = 'cuda' if torch.cuda.is_available() else 'cpu'
lr = 0.1
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(1, 1)).to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)
loss_fn = nn.MSELoss(reduction='mean')
train_step_fn = make_train_step(model, loss_fn, optimizer)
val_step_fn = make_val_step_fn(model, loss_fn)
writer = SummaryWriter('runs/simple_linear_regression')
x_dummy, y_dummy = next(iter(train_loader))
writer.add_graph(model, dummy_x.to(device))

Overwriting model_configuration/v3.py


In [105]:
%run -i model_configuration/v3.py

In [106]:
%%writefile model_training/v5.py
n_epochs = 200
losses = []
val_losses = []

for epoch in range(n_epochs):
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)

    # validation - no gradients in validation
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
        val_losses.append(val_loss)

    writer.add_scalars(main_tag='loss',
                    tag_scalar_dict={
                        'training': loss,
                        'validation': val_loss
                    },
                    global_step=epoch)
writer.close()

Writing model_training/v5.py


In [107]:
%run -i model_training/v5.py

In [108]:
print(model.state_dict())

OrderedDict([('0.weight', tensor([[1.9583]])), ('0.bias', tensor([1.0099]))])


In [110]:
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 7086), started 0:32:58 ago. (Use '!kill 7086' to kill it.)

# Saving and Loading Models

## saving

In [111]:
checkpoint = {'epoch': n_epochs,
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict(),
              'loss': losses,
              'val_loss': val_losses}
torch.save(checkpoint, 'model_checkpoint.pth')

## Resuming Training

In [112]:
%run -i data_preparation/v2.py
%run -i model_configuration/v3.py

In [113]:
print(model.state_dict())

OrderedDict([('0.weight', tensor([[0.7645]])), ('0.bias', tensor([0.8300]))])


In [116]:
checkpoint = torch.load('model_checkpoint.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

saved_epoch = checkpoint['epoch']
saved_losses = checkpoint['loss']
saved_val_losses = checkpoint['val_loss']

model.train()

Sequential(
  (0): Linear(in_features=1, out_features=1, bias=True)
)

In [117]:
print(model.state_dict())

OrderedDict([('0.weight', tensor([[1.9583]])), ('0.bias', tensor([1.0099]))])


In [118]:
print(model.state_dict())

OrderedDict([('0.weight', tensor([[1.9583]])), ('0.bias', tensor([1.0099]))])


In [119]:
%run -i model_configuration/v3.py

In [122]:
checkpoint = torch.load('model_checkpoint.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(model.state_dict())

OrderedDict([('0.weight', tensor([[1.9583]])), ('0.bias', tensor([1.0099]))])


In [123]:
new_inputs = torch.tensor([[.20], [.34], [.57]])
model.eval()
model(new_inputs.to(device))

tensor([[1.4015],
        [1.6757],
        [2.1261]], grad_fn=<AddmmBackward0>)

# Chapter 2.1

In [124]:
import numpy as np
import datetime

import torch
import torch.optim as optim
import torch.nn as nn
import torch.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('fivethirtyeight')

## Going Classy

In [125]:
class StepByStep(object):
    pass

In [129]:
class StepByStep(object):
    def __init__(self, model, loss_fn, optimizer):
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        self.train_loader = None
        self.val_loader = None
        self.writer = None
        self.losses = []
        self.val_losses = []
        self.total_epochs = 0
        self.train_step_fn = self._make_train_step_fn()
        self.val_step_fn = self._make_val_step_fn()

    def to(self, device):
        try:
            self.device = device
            self.model.to(device)
        except RuntimeError:
            self.device = ('cuda' if torch.cuda.is_available() else 'cpu')
            print(f"Couldn't send it to {device}, sending it to {self.device} instead")
            self.model.to(self.device)

    def set_loaders(self, train_loader, val_loader=None):
        self.train_loader = train_loader
        self.val_loader = val_loader

    def set_tensorboard(self, name, folder='runs'):
        suffix = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
        self.writer = SummaryWriter(f'{folder}/{name}_{suffix}')

In [130]:
def _make_train_step_fn(self):
    def perform_train_step_fn(x, y):
        self.model.train()
        yhat = self.model(x)
        loss = self.loss_fn(yhat, y)
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        return loss.item()
    return perform_train_step_fn

def _make_val_fn(self):
    def perform_val_step_fn(x, y):
        self.model.eval()
        yhat = self.model(x)
        loss = self.loss_fn(yhat, y)
        return loss.item()
    return perform_val_step_fn